# Building conversational applications with Strands Agents

In this notebook, you learn how to use Strands agents to integrate external capabilities into conversational applications.

This is a conversion of the original module9-converse_with_tools.ipynb that used the Converse API with tool specifications. Instead, we'll use Strands agents for a more streamlined approach.

## Environment setup

In this task, you set up your environment and install the required packages for Strands agents.

In [1]:
from strands import Agent
from strands.models.bedrock import BedrockModel
from mcp import stdio_client, StdioServerParameters
from strands import Agent
from strands.tools.mcp import MCPClient

model_id = "amazon.nova-lite-v1:0"

## Define the MCP Server command

In [2]:
# Define the MCP server using stdio transport
stdio_mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="uv", 
        args=["run", "module9-mcp_server.py"]
    )
))

## Instantiate the MCP Client and create the Strands Agent

Now we create a Strands agent that uses the Bedrock LLM and our defined tools. The agent will automatically handle tool calling and conversation flow.

In [3]:
import sys
from pathlib import Path

from strands import Agent
from strands.models.bedrock import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import stdio_client, StdioServerParameters

model_id = "amazon.nova-lite-v1:0"

server_file = Path.cwd() / "module9-mcp_server.py"

if not server_file.exists():
    raise FileNotFoundError(f"Cannot find MCP server file: {server_file}")

mcp_errlog = open("mcp-server-error.log", "w", encoding="utf-8")

stdio_mcp_client = MCPClient(
    lambda: stdio_client(
        StdioServerParameters(
            command=sys.executable,
            args=[str(server_file)],
            cwd=str(Path.cwd())
        ),
        errlog=mcp_errlog
    )
)

In [4]:
# Create the Bedrock LLM wrapper for Strands
llm = BedrockModel(
    model_id=model_id
)

# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    tools = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    agent = Agent(
        model=llm,
        tools=tools,
        system_prompt="You are a helpful assistant that can perform calculations and search for information on DuckDuckGo."
    )

    # Send the question
    agent("What is Amazon SageMaker? What is launch year multiplied by 2")

<thinking>The User has asked two questions: one is about a specific service, Amazon SageMaker, and the other is a simple arithmetic operation involving the year Amazon SageMaker was launched. I can answer the arithmetic operation directly but need to search for the launch year of Amazon SageMaker to complete the second part of the question.</thinking>

Tool #1: duckduckgo_search
<thinking>I found the launch year of Amazon SageMaker, which is November 2017. Now, I will calculate the launch year multiplied by 2.</thinking> 
Tool #2: calculator
<thinking>I have completed the arithmetic operation and found that the launch year of Amazon SageMaker, 2017, multiplied by 2 equals 4034.</thinking>

Here is the information you asked for:
- **Amazon SageMaker**: It is a cloud-based machine-learning platform that allows developers to create, train, and deploy machine-learning models on the cloud. It can also be used to deploy models on embedded systems and edge devices.
- **Launch year multiplied 